# Day I - C) Error estimation and mesh refinement

Using a simple 2D inductor computation, we will illustrate the influence of the mesh on the accuracy of the solution, as well as classical error estimators used to drive adaptive mesh refinement.

- **<font color='green'>[RUN & OBSERVE]</font>** $\rightarrow$ the cell should be run directly without modification
- **<font color='orange'>[RUN & PLAY]</font>** $\rightarrow$ the cell can be run directly, but some parameters should be changed interactively
- **<font color='red'>[FILL & RUN]</font>**    $\rightarrow$ the cell should be filled before being run
- **<font color='magenta'>[FILL & PLAY]</font>** $\rightarrow$ the cell should be filled, and then some parameters should be changed interactively.

|**<u>Objective</u>** : Understand error estimation and adaptive mesh refinement|
|---|
|*Indicative duration* : 45 minutes|

## Outlines:

<a id='outlines'></a>

- [**1) Magnetostatic computation**](#1_magnetostatic) (*10 minutes*)
- [**2) Energy**](#2_energy) (*15 minutes*)
    - [a) Expressions](#2a_expressions)
    - [b) Evolution with the mesh size](#2b_evolution_mesh_size)
- [**3) Adaptative h-refinement**](#3_adaptive_h_refinement) (*20 minutes*)
    - [a) Some preliminaries on de Rham complexes](#3a_de_rham_complexes)
    - [b) Simple recovery based estimator](#3b_recovery_estimator)
    - [c) Residual and equilibrated estimator](#3c_residual_estimator)
    - [d) Adaptive mesh refinement](#3d_adaptive_mesh_refinement)
    - [e) Comparison uniform and adaptive refinement](#3e_comparison_refinement)

<a id='1_magnetostatic'></a>

---

## 1) Magnetostatic computation

|*Indicative duration* : 10 minutes|
|---|

<a id='1_magnetostatic'></a>

First we define the geometry and mesh.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
import ngsolve
from utils.myGeometries import gaped_inductor
from ngsolve.webgui import Draw

mesh = gaped_inductor(airgap = 0, h = 0.01)
print(f"Faces: {mesh.GetBoundaries()}")
print(f"Regions: {mesh.GetMaterials()}")
Draw(mesh)

Then we encapsulate the magnetostatic solver. Solving the weak formulation of the magnetostatic problem consist find $a\in H^1_0 (\Omega)$ such that

$$ \forall v\in H^1_0(\Omega) , \quad \int_\Omega  \nu  \;\text{curl}~ a_z \cdot \text{curl}~ v  = \int_\Omega v j_z, $$

with, in our 2D case: $\text{curl}(\cdot) =  \begin{pmatrix} 0 & 1 \\-1 & 0 \end{pmatrix} \text{grad}(\cdot)$


We recall the NGSolve syntax:
- `a += u * v * dx("coilP")` will integrate the form on `coilP` domain
- `a += u * v * dx("coilP|coilN")` will integrate the form on the union of `coilP` and `coilN` domains

|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
# configuration of physical parameters
# ---------------------------------------------------------------
mu_r = 1000      # relative permeability
J = 10e6         # source current density A/m^2
from ngsolve import pi
mu0 = 4e-7 * pi  # H/m, void magnetic permeability
# ---------------------------------------------------------------

from ngsolve import H1, CF, GridFunction, grad, dx, LinearForm, BilinearForm
from time import time

def curl(a):
    return CF(((0,1),(-1,0)), dims = (2,2)) * grad(a)

def solve_magnetostatic(mesh, order = 1):
    """ Solve linear magnetostatic problem """
    # finite element space
    t0 = time()
    fes = H1(mesh, order = order, dirichlet = "out")
    
     # variational formulation
    a, v = fes.TnT()
    A = BilinearForm(fes)
    A += .....................
    l = LinearForm(fes)
    l += .....................
    
    # Assemble matrix and solver 
    K = A.Assemble().mat
    f = l.Assemble().vec
    sol = GridFunction(fes)
    sol.vec.data = K.Inverse(freedofs = fes.FreeDofs()) * f
    print(f"Solution computed in {time()-t0 :.3f} s ({mesh.nv} DoFs)")

    return sol

# small test on a coarse mesh
sol = solve_magnetostatic(mesh)
Draw(sol)

We compute the magnetostatic solution on several mesh, more and more refined.

**<font color='red'> Q1. Plot the meshing and computation time with respect to the mesh size. </font>**


|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
# Run with different meshes
hList = [0.005, 0.0025, 0.001, 0.0005, 0.00025]
solList = [solve_magnetostatic(gaped_inductor(airgap = 0, h = h))for h in hList]

In [ ]:
# Display
meshing_time = [.....................]
solve_time = [.....................]
n_elts = np.array([.....................])       # number of mesh elements
plt.loglog(n_elts, meshing_time, label = "mesh")
plt.loglog(n_elts, solve_time, label = "solve")
plt.loglog(n_elts, n_elts/n_elts[0]*solve_time[0], '--', label = "n")

plt.legend()
plt.xlabel("ne"); plt.ylabel("Time (s)"); plt.grid()
plt.show()

---

## 2) Energy

|*Indicative duration* : 15 minutes|
|---|

<a id='2_energy'></a>


### 2.a) Expressions

<a id='2a_expressions'></a>

To have an idea of how close we are to the real solution, we compute indicators, such as 

- Stored magnetic energy:

$$ E_{mag} = \int_\Omega \frac{|\text{curl}\;a|^2}{2 \mu} \; \mathrm{d}x$$

- Total energy:

$$ E_{tot} = \int_\Omega \frac{|\text{curl}\;a|^2}{2 \mu} - j \cdot a \; \mathrm{d}x $$

**<font color='red'> Q2. Justify that the closest to the exact solution, the lower the total energy. </font>**

|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
from ngsolve import CF, Integrate


def magnetic_energy_density(a):
    B = curl(a)
    mesh = a.space.mesh
    return mesh.MaterialCF({"iron":..................... },  # energy density in iron
                           default = .....................)  # energy density elsewhere
    
def magnetic_energy(a):
    return Integrate( magnetic_energy_density(a), mesh)  

def total_energy_density(a):
    B = curl(a)
    mesh = a.space.mesh
    return mesh.MaterialCF({"iron":  .....................,  # total energy density in iron
                            "condP": .....................,  # total energy density in positive conductor
                            "condN": .....................}, # total energy density in negative conductor
                           default = .....................)  # total energy elsewhere
    
def total_energy(a):
    return Integrate( total_energy_density(a), mesh)

We can compute these indicators on the previous solutions with various discretizations. We also have access to the reference ones, precomputed on a very fine mesh and higher polynomial order.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
# Compute the magnetostatic energy stored
energyList = [magnetic_energy(sol) for sol in solList] 
totalEnergyList = [total_energy(sol) for sol in solList] 

# Reference values computed on a fine mesh and higher polynomial order
ref_mag_energy = 15.411136624676304
ref_tot_energy = -15.44279963340253

### 2.b) Evolution with the mesh size

<a id='2b_evolution_mesh_size'></a>

Smaller elements lead to smaller error. Observe the evolution of total energy and maximization of magnetic energy.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
# Display

import matplotlib.pyplot as plt

fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('h')
ax1.set_ylabel('Magnetic Energy (J)', color=color)
ax1.hlines(ref_mag_energy, min(hList), max(hList), color=color, linestyles = '--')
ax1.semilogx(hList, energyList, ':o', color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

color = 'tab:blue'
ax2.set_ylabel('Total Energy (J)', color=color)
ax2.semilogx(hList, totalEnergyList, ':s')
ax2.hlines(ref_tot_energy, min(hList), max(hList), color=color, linestyles = '--')
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.show()

_______
## 3) Adaptative h-refinement

|*Indicative duration* : 20 minutes|
|---|

<a id='3_adaptive_h_refinement'></a>

In the following, we try to illustrate the idea of a posteriori error estimation.
This information can be use to refine the mesh locally, in order to improve accuracy while keeping a reasonnable computation time.

### 3.a) Some preliminaries on de Rham complexes

<a id='3a_de_rham_complexes'></a>

See [this webpage](https://docu.ngsolve.org/v6.2.2404/i-tutorials/unit-2.3-hcurlhdiv/hcurlhdiv.html).

We have seen $H^1$ space, approximated by continuous nodal functions, as well as $L^2$ space approximate by discontinuous element-wise functions.

For vector fields, there are intermediate levels of continuity:
- **Continuity of tangential component** (as for $H$): generally we take their curl, so we define

$$ H(\text{curl}) = \{v \in L^2,  \text{curl}(v) \in L^2 \} $$

$H(\text{curl})$ is approximated using edge-element, the DoFs represent the elementary circulations.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
from ngsolve import HCurl

uHCurl = GridFunction(HCurl(mesh))
uHCurl.vec.data[-1] = 1

Draw(uHCurl, vectors={"grid_size" : 50})

- **Continuity of normal component** (as for $B$): generally we take their div, so

$$ H(\text{div}) = \{v \in L^2,  \text{div}(v) \in L^2 \} $$


$H(\text{div})$ is approximated using facet-element, the DoFs represent the elementary fluxes.


|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
from ngsolve import HDiv

uHDiv = GridFunction(HDiv(mesh))
uHDiv.vec.data[-1] = 1e-3

Draw(uHDiv, vectors={"grid_size" : 50})

This kind of elements will be very important later: chosing nonconforming element can indeed lead to [wrong results](https://web.pdx.edu/~gjay/pub/MaxwellGoodBadUgly.html)! 

This will be of the first importance for 3D simulations (see next day)!


### 3.b) Simple recovery based estimator

<a id='3b_recovery_estimator'></a>



A first and simple idea is to evaluate the difference between the computed $H$ and its projection to the function space $H(\text{Curl}, \Omega)$ that ensures tangential continuity by construction.

The difference is related to the tangential jumps of $H$, which are due to discretization error.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
from ngsolve import HCurl, Norm, VOL

def ZZlike_estimate(sol):
    """ Recovery-based (Zienkiewicz-Zhu like) estimator """
    # measure the jump of tangiential components of h
    # heuristic and no garantee... But simpler!
    mesh = sol.space.mesh
    mu = mesh.MaterialCF({"iron" : mu0*mu_r}, default = mu0)
    H_ref = curl(sol) / mu    
    H_hcurl = GridFunction(HCurl(mesh, order = sol.space.globalorder ))
    H_hcurl.Set(H_ref)
    return (H_ref - H_hcurl)**2 * mu

ZZestimateList = [ZZlike_estimate(sol) for sol in solList]
maxZZestimateList = [max(Integrate(err, sol.space.mesh, VOL, element_wise=True)) for err, sol in zip(ZZestimateList, solList)]

In [ ]:
# Display
index = 2
Draw(ZZestimateList[index], solList[index].space.mesh)

### 3.c) Residual and equilibrated estimator

<a id='3c_residual_estimator'></a>

A more elaborated and theoretically sound error estimator is to account for the residual error as well as the true tangential jumps, without relying on a heuristic projection.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
from ngsolve import specialcf, L2

def BRV_estimate(sol):
    """ 
    Residual-based a posteriori error estimator (Babuška–Rheinboldt / Verfürth type)
    for 2D magnetostatics in the vector potential formulation.

    Model problem:   curl( nu * curl A ) = J      (nu = 1/mu = reluctivity)

    For every triangle K it computes the indicator

        eta_K^2 =  (h_K^2 / nu)  * || J ||^2_K                      <- element residual
                 + sum_{F in dK} (h_F / (2 nu_F)) * || [H x n] ||^2_F   <- facet jumps

    The global estimate is  eta = sqrt( sum_K eta_K^2 ).
    Reliable (proven upper bound, up to constants) but a bit tedious to implement.

    Returns: a piecewise-constant GridFunction holding eta_K^2 for each element.
    """
    mesh = sol.space.mesh

    # --- ial and fields -------------------------------------------------
    nu_cf = 1/mesh.MaterialCF({"iron" : mu0*mu_r}, default = mu0)
    B = curl(sol)
    H = B * nu_cf

    # --- Geometric quantities -------------------------------------------------
    n = specialcf.normal(2)        # unit normal on element facets (edges in 2D)
    h_elem = specialcf.mesh_size   # local element size h_K
    # Facet size h_F or  element size, which is equivalent in 2D up to a constant factor.
    h_facet = specialcf.netgen_facetsize if hasattr(specialcf, "netgen_facetsize") else specialcf.mesh_size

    # --- Jump of the tangential component of H --------------------------------
    # Physics: the tangential component of H must be continuous across an
    # interface (if there is no surface current). A discrete solution only
    # satisfies this approximately, so the jump measures the error.
    # In 2D the tangential component is  H . t = H_x*n_y - H_y*n_x  (t = n rotated by 90°).
    # H.Other() is the value of H seen from the neighbouring element on the facet.
    H_other = H.Other()
    n_tangent_jump = (H[0] * n[1] - H[1] * n[0]) - (H_other[0] * n[1] - H_other[1] * n[0])

    # --- Space for the indicators ---------------------------------------------
    # One number per element: L2 of order 0 = piecewise constants.
    # Testing a linear form with v_l2 (=1 on K, 0 elsewhere) gives the
    # integral over K, i.e. exactly the local contribution eta_K^2.
    fes_L2 = L2(mesh, order=0)
    u_l2, v_l2 = fes_L2.TnT()

    lf = LinearForm(fes_L2)

    # --- Term 1: element residual ---------------------------------------------
    # Interior residual is  J - curl(H). Here only the data term J is kept,
    # because for low-order elements curl(H) = 0 inside each triangle.
    # With higher-order elements we should add the curl(H) contribution.
    # Only integrated on the coil regions ("condP|condN" = positive/negative
    # conductors), since J = 0 elsewhere.
    # Weight h_K^2 / nu = h_K^2 * mu is consistent with the energy norm.
    lf += (h_elem ** 2)/nu_cf * (J ** 2) * v_l2 * dx("condP|condN")
    
    # --- Term 2: facet jump residual -------------------------------------------
    # Average reluctivity on the two sides of the facet, so the weight
    # stays meaningful at iron/air interfaces where nu jumps strongly.
    nu_avg = 0.5 * (nu_cf + nu_cf.Other())
    # dx(skeleton=True) integrates over all interior facets, each visited once.
    # Testing with (v_l2 + v_l2.Other()) adds the contribution to BOTH neighbouring
    # elements, and the factor 0.5 splits the facet term equally between them.
    lf += 0.5 * h_facet / nu_avg * (n_tangent_jump ** 2) * (v_l2 + v_l2.Other()) * dx(skeleton=True)

    # Assemble: lf.vec[i] now holds eta_K^2 for element i.
    lf.Assemble()

    # --- Output ----------------------------------------------------------------
    # Copy the vector into a GridFunction so it can be drawn and evaluated.
    # Values are eta_K^2. 
    eta = GridFunction(fes_L2)
    eta.vec.data = lf.vec

    return eta

BRVestimateList = [BRV_estimate(sol) for sol in solList]
maxBRVestimateList = [max(Integrate(err, sol.space.mesh, VOL, element_wise=True)) for err, sol in zip(BRVestimateList, solList)]

In [ ]:
# Display
index = 2
Draw(BRVestimateList[index], solList[index].space.mesh)

### 3.d) Adaptive mesh refinement

<a id='3d_adaptive_mesh_refinement'></a>

Dolfer marking : 
SOLVE→ESTIMATE→MARK→REFINE→SOLVE→...

we start with a very coarse mesh.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
mesh = gaped_inductor(airgap = 0, h = 1)
Draw(mesh)

Then we solve the problem and estimate the local error...

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
estimate = BRV_estimate # ZZlike_estimate
toterr_uniform = [Integrate(estimate(sol), mesh) for sol in solList]
maxerr_uniform = [max(Integrate(estimate(sol), mesh, VOL, element_wise = True)) for sol in solList]

sol = solve_magnetostatic(mesh)
err = estimate(sol)
Draw(err, sol.space.mesh)

We mark the elements to refine... (Lets set a threshold at 25% of the max error).

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
import numpy as np

threshold = 0.25

eta2 = Integrate(err, mesh, VOL, element_wise=True)
maxerr_adapt = [max(eta2)]
toterr_adapt = [np.sum(eta2)]
n_elts_adapt = [mesh.ne]
mesh.ngmesh.Elements2D().NumPy()["refine"] = eta2.NumPy() > threshold*maxerr_adapt[-1]

mesh.Refine()
Draw(mesh)

And we loop!
|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
for i in range(8):
    sol = solve_magnetostatic(mesh)
    err = estimate(sol)
    eta2 = Integrate(err, mesh, VOL, element_wise=True)
    maxerr_adapt.append(max(eta2))
    toterr_adapt.append(np.sum(eta2))
    n_elts_adapt.append(mesh.ne)
    mesh.ngmesh.Elements2D().NumPy()["refine"] = eta2.NumPy() > threshold*maxerr_adapt[-1]
    mesh.Refine()

In [ ]:
# Display the final mesh and solution
Draw(curl(sol), sol.space.mesh)

### 3.e) Comparison uniform and adaptive refinement

<a id='3e_comparison_refinement'></a>

Adaptive refinement is far more efficient than uniform refinement. As a rule of thumb, we have to refine near the corners.

**<font color='red'> Q3. Discuss computation time of adaptive refinement versus a computation made on an already fine mesh.</font>**



|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
# display
n_elts_adapt = np.array(n_elts_adapt)
n_elts_uniform = np.array([sol.space.mesh.ne for sol in solList])
plt.loglog(n_elts_uniform, toterr_uniform, ':o', label = "total error (uniform)")
plt.loglog(n_elts_adapt, toterr_adapt, 'o:', label = "total error (adaptive)")
plt.loglog(n_elts_uniform, maxerr_uniform, ':o', label = "max local error (uniform)")
plt.loglog(n_elts_adapt, maxerr_adapt, 'o:', label = "max local error (adaptive)")
plt.loglog(n_elts_uniform, toterr_uniform[0]/(n_elts_uniform/n_elts_uniform[0]), '--', label = "n")
plt.loglog(n_elts_adapt, toterr_adapt[0]/(n_elts_adapt/n_elts_adapt[0])**2, '--', label = "$n^2$")

plt.legend(); plt.grid(); plt.xlabel("n");  plt.ylabel("estimated max(error)")
plt.show()

**<font color='red'> Q4. Experiment with the two error estimators, different thresholds and refinement iterations.</font>**

---
## References:

- Z. Tang, Y. Le Menach, E. Creusé, S. Nicaise, F. Piriou, and N. Nemitz, “Residual and equilibrated error estimators for magnetostatic problems solved by finite element method,” IEEE Transactions on Magnetics, vol. 49, pp. 5715–5723, Dec 2013. DOI [10.1109/TMAG.2013.2271993](https://doi.org/10.1109/TMAG.2013.2271993)

- Z. Tang, Chapter - A posteriori error estimates for low frequency electromagnetic computations,   Editor(s): Franz Chouly, Stéphane P.A. Bordas, Roland Becker, Pascal Omnes, Advances in Applied Mechanics, Elsevier, Volume 60, ISBN 9780443294525, June 2025. DOI [10.1016/bs.aams.2025.02.007](https://doi.org/10.1016/bs.aams.2025.02.007)

- Z. Tang, On a posteriori error estimation in low-frequency computational electromagnetics: reflections on historical foundations, current realities. International Compumag Society (ICS) newsletter, Vol. 33, N . 3, 11/2026

- O.C. Zienkiewicz & J.Z. Zhu, "A simple error estimator and adaptive procedure for practical engineering analysis", IJNME 24, 337–355, (1987) DOI [10.1002/nme.1620240206](www.doi.org/10.1002/nme.1620240206)

[*Back to top - outlines*](#outlines)